In [ ]:
import sys; sys.path.insert(0, "..")
from pathlib import Path
from src.aef_fetcher import fetch_embeddings
from src.classifier import load_model
from src.inference import run_inference
from src.verdict import generate_verdict

MODEL_DIR = Path("../models")
OUT_DIR = Path("../data/outputs/test_parcel")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Small parcel in Karnataka (plantation likely present)
PARCEL = {
    "type": "Polygon",
    "coordinates": [[[77.5, 12.9], [77.51, 12.9], [77.51, 12.91], [77.5, 12.91], [77.5, 12.9]]]
}

In [ ]:
ds = fetch_embeddings(PARCEL, years=list(range(2017, 2026)))
print(f"Fetched: {ds['embeddings'].shape}")

In [ ]:
model, class_map, _ = load_model(MODEL_DIR)
inference_paths = run_inference(ds, model, class_map, OUT_DIR)
print(f"Inference complete: {list(inference_paths.keys())}")

In [ ]:
verdict = generate_verdict("test_karnataka", inference_paths, PARCEL, OUT_DIR)
import json; print(json.dumps(verdict, indent=2))

In [ ]:
import matplotlib.pyplot as plt
import rasterio
years = sorted(inference_paths.keys())
fig, axes = plt.subplots(3, 3, figsize=(12, 12))
for ax, year in zip(axes.flat, years):
    with rasterio.open(inference_paths[year]["lulc"]) as src:
        ax.imshow(src.read(1), vmin=0, vmax=6, cmap="tab10")
        ax.set_title(year)
        ax.axis("off")
plt.tight_layout(); plt.show()